# Q5 - Codabench Submission (MIND + EB-NeRD)

Generates a `prediction.txt`/`predictions.txt` (zipped) submission per
dataset, reusing the same `score_inview` adapters as Q4 -- a leaderboard
submission is exactly Q4's re-ranking framing (rank each impression's own
`article_ids_inview`), not Q2/Q3's full-catalog retrieval.

- **MIND** (`codabench.org/competitions/13967`): format confirmed directly
  from the competition's own Submission Guidelines page (`prediction.txt`,
  singular). Per SPEC.md Q5 #3, no separate hidden test file exists on the
  Data tab, so this predicts over MINDsmall's provider `dev/` -- this
  project's own `test` split.
- **EB-NeRD** (`codabench.org/competitions/2469`): format inferred from the
  RecSys 2024 Challenge's own starter repo utility
  (`ebrec.utils._python.write_submission_file`, `predictions.txt`, plural)
  -- **not yet directly confirmed** against the competition's own
  Submission Guidelines page (see SPEC.md Q5 #2). Same population caveat as
  MIND applies here too and is *not yet* checked: this predicts over
  EB-NeRD's own `test` split (provider `validation/`), which may or may not
  be what the competition's hidden ground truth actually covers -- MIND's
  real submission already failed once on exactly this kind of assumption
  (see SPEC.md Q5 #3's `MINDlarge_test` finding), so treat this dataset's
  output as unverified until confirmed on Codabench.
- **EB-NeRD small/large** (`ebnerd_small`/`ebnerd_large`): generated locally
  for completeness (larger article catalogs and user bases than the demo
  bundle -- see `src/build_pipeline.ipynb`), but neither is a separate
  Codabench track. The real EB-NeRD competition scores `ebnerd_testset.zip`,
  not `ebnerd_small`/`ebnerd_large` (see SPEC.md Q5 #3) -- do not submit
  these zips to Codabench.
- **MIND large** (`mind_large`): built from the provider's
  `MINDlarge_train`/`MINDlarge_dev` bundles, not the real held-out
  `MINDlarge_test` -- same non-submittable caveat as `ebnerd_small`/
  `ebnerd_large` above.

Run top-to-bottom (or via `python generate_predictions.py`) to rebuild
`submissions/{dataset}/*.zip`.

Loading/filtering uses `polars` throughout (same reasoning as
`src/build_pipeline.ipynb`/`src/bm25_retrieval.ipynb`), with the same
`BUILD_LARGE_ONLY` flag convention and progress appended to
`build_progress.log`. `behaviors.parquet` is read via a lazy
`scan_parquet().filter().select()` since this notebook only ever needs the
`test` split -- at `ebnerd_large` scale that skips materializing ~12M
train/val rows entirely.

## Setup

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import zipfile

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.bm25 import tokenize, build_index, get_scores
from cs4406m26_assignment1c1.embeddings import mean_pool, cosine_similarity_subset, normalize_rows


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
SUBMISSIONS_DIR = ROOT / "submissions"
PROGRESS_LOG = ROOT / "build_progress.log"

# Same flag/convention as src/build_pipeline.ipynb and src/bm25_retrieval.ipynb.
BUILD_LARGE_ONLY = True
DATASETS = (
    ["ebnerd_large", "mind_large"]
    if BUILD_LARGE_ONLY
    else ["ebnerd", "ebnerd_small", "mind", "ebnerd_large", "mind_large"]
)
RECENT_N_CLICKS = 20
SPLIT = "test"  # MIND == MINDsmall's provider dev/; EB-NeRD == provider validation/

# Per-competition submission filename -- MIND confirmed directly from its own
# Submission Guidelines page (singular); EB-NeRD inferred from its starter
# repo's write_submission_file default (plural) -- see SPEC.md Q5 #2.
# Neither ebnerd_small nor ebnerd_large/mind_large is its own separate
# Codabench track (the real competitions score ebnerd_testset.zip and
# MINDlarge_test respectively, not any of these -- see SPEC.md Q5 #3), so
# they reuse their family's filename convention for local-format
# consistency only -- do not submit these zips.
TXT_FILENAME = {
    "mind": "prediction.txt",
    "ebnerd": "predictions.txt",
    "ebnerd_small": "predictions.txt",
    "ebnerd_large": "predictions.txt",
    "mind_large": "prediction.txt",
}

# Q4 found embeddings ranking better than BM25 on every dataset's test split
# (see data/processed/{dataset}/eval_metrics.json for exact AUC figures) --
# generate that one for the real submission, plus BM25 for comparison.
METHODS = ["embedding", "bm25"]


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] {message}\n")
        f.flush()


log_progress(f"generate_predictions started (BUILD_LARGE_ONLY={BUILD_LARGE_ONLY}, datasets={DATASETS})")

feature_store = {}
embeddings_raw = {}
bm25_index = {}
title_by_id = {}
corpus = {}

for name in DATASETS:
    feature_store[name] = {
        "articles": pl.read_parquet(DATA_DIR / name / "articles.parquet", columns=["article_id", "title", "abstract"]),
        # Only the SPLIT ("test") rows are ever used in this notebook -- filter
        # pushed down at scan time so ebnerd_large's other ~12M train/val rows
        # are never materialized in memory at all.
        "behaviors": (
            pl.scan_parquet(DATA_DIR / name / "behaviors.parquet")
            .filter(pl.col("split") == SPLIT)
            .select(["impression_id", "user_id", "article_ids_inview"])
            .collect()
        ),
        "history": pl.read_parquet(DATA_DIR / name / "history.parquet", columns=["user_id", "article_id_sequence"]),
    }

    embedding_path = DATA_DIR / name / "article_embeddings.parquet"
    if not embedding_path.exists():
        raise FileNotFoundError(
            f"missing {embedding_path}. Run src/compute_embeddings_kaggle.ipynb on Kaggle first (see README.md)."
        )
    embeddings_raw[name] = pl.read_parquet(embedding_path)

    # Rebuild the BM25 index locally (cheap, seconds) -- same pattern as Q4.
    articles = feature_store[name]["articles"]
    texts = (articles["title"].fill_null("") + " " + articles["abstract"].fill_null("")).to_list()
    doc_tokens = [tokenize(t) for t in texts]
    bm25_index[name] = build_index(articles["article_id"].to_list(), doc_tokens)
    title_by_id[name] = dict(zip(articles["article_id"].to_list(), articles["title"].to_list()))

    # Corpus embedding matrix, reindexed to articles.parquet order -- same pattern as Q3/Q4.
    emb_lookup = dict(zip(
        embeddings_raw[name]["article_id"].to_list(),
        [np.asarray(v) for v in embeddings_raw[name]["embedding"].to_list()],
    ))
    doc_ids = articles["article_id"].to_numpy()
    missing_ids = set(doc_ids) - set(emb_lookup)
    if missing_ids:
        raise ValueError(f"{name}: {len(missing_ids)} articles have no embedding")
    emb_matrix = np.stack([emb_lookup[aid] for aid in doc_ids]).astype(np.float32)
    corpus[name] = {"doc_ids": doc_ids, "matrix": emb_matrix, "embedding_lookup": emb_lookup}
    log_progress(
        f"  {name}: setup complete ({bm25_index[name].n_docs} docs, "
        f"{feature_store[name]['behaviors'].height} {SPLIT} impressions)"
    )

{name: {"bm25_docs": bm25_index[name].n_docs, "embedding_docs": len(corpus[name]["doc_ids"]),
        "test_impressions": feature_store[name]["behaviors"].height}
 for name in DATASETS}

{'ebnerd_large': {'bm25_docs': 125541,
  'embedding_docs': 125541,
  'test_impressions': 12566385},
 'mind_large': {'bm25_docs': 104151,
  'embedding_docs': 104151,
  'test_impressions': 376471}}

In [2]:
def test_setup_aligned():
    for name in DATASETS:
        articles = feature_store[name]["articles"]
        assert bm25_index[name].n_docs == len(articles)
        assert corpus[name]["matrix"].shape[0] == len(articles)
        assert (corpus[name]["doc_ids"] == articles["article_id"].to_numpy()).all()
        assert not np.isnan(corpus[name]["matrix"]).any()


test_setup_aligned()
print("ok: BM25 indexes and embedding matrices rebuilt/loaded and aligned with articles.parquet, every dataset")

ok: BM25 indexes and embedding matrices rebuilt/loaded and aligned with articles.parquet, every dataset


## score_inview adapters

Identical to Q4's adapters (`score_inview(user_id, article_ids_inview) ->
dict[article_id, float]`, BM25 memoizing the last user's full-corpus score
vector), one instance per dataset. Cold-start users get an all-zero score
vector -- a well-defined, fully-tied ranking rather than a crash (see Q4's
`evaluation_harness.ipynb` for why this is safe).

In [3]:
def build_user_query_tokens(article_id_sequence, title_lookup: dict, recent_n: int = RECENT_N_CLICKS) -> list[str]:
    recent_ids = list(article_id_sequence)[-recent_n:]
    titles = [title_lookup.get(aid, "") for aid in recent_ids]
    return tokenize(" ".join(t for t in titles if t))


def build_user_query_vector(article_id_sequence, embedding_lookup: dict, recent_n: int = RECENT_N_CLICKS):
    recent_ids = list(article_id_sequence)[-recent_n:]
    return mean_pool(recent_ids, embedding_lookup)


def make_score_inview_adapters(dataset: str) -> dict:
    id_to_idx = {aid: i for i, aid in enumerate(bm25_index[dataset].doc_ids)}
    history = feature_store[dataset]["history"]
    history_lookup = dict(zip(history["user_id"].to_list(), history["article_id_sequence"].to_list()))
    title_lookup = title_by_id[dataset]
    embedding_lookup = corpus[dataset]["embedding_lookup"]
    emb_matrix = corpus[dataset]["matrix"]
    emb_doc_ids = corpus[dataset]["doc_ids"]
    # Precomputed once per dataset, not per impression -- see Q4's
    # evaluation_harness.ipynb / embeddings.py's cosine_similarity_subset
    # docstring for why rebuilding this per call was the dominant cost of
    # the whole embedding-scoring pass at ebnerd_large scale.
    emb_id_to_idx = {aid: i for i, aid in enumerate(emb_doc_ids)}
    emb_corpus_unit = normalize_rows(emb_matrix.astype(np.float32))

    bm25_cache = {"user_id": None, "scores": None}

    def bm25_fn(user_id, article_ids_inview):
        if bm25_cache["user_id"] != user_id:
            seq = history_lookup.get(user_id, [])
            query_tokens = build_user_query_tokens(seq, title_lookup)
            bm25_cache["user_id"] = user_id
            bm25_cache["scores"] = get_scores(bm25_index[dataset], query_tokens)
        scores = bm25_cache["scores"]
        return {aid: float(scores[id_to_idx[aid]]) for aid in article_ids_inview}

    def embedding_fn(user_id, article_ids_inview):
        seq = history_lookup.get(user_id, [])
        query_vector = build_user_query_vector(seq, embedding_lookup)
        scored = cosine_similarity_subset(query_vector, emb_corpus_unit, emb_doc_ids, emb_id_to_idx, article_ids_inview)
        return {aid: scored.get(aid, 0.0) for aid in article_ids_inview}

    return {"bm25": bm25_fn, "embedding": embedding_fn}


score_inview_adapters = {name: make_score_inview_adapters(name) for name in DATASETS}

In [4]:
def test_score_inview_adapters():
    for name in DATASETS:
        sample = feature_store[name]["behaviors"].row(0, named=True)
        inview = list(sample["article_ids_inview"])
        for method in METHODS:
            scored = score_inview_adapters[name][method](sample["user_id"], inview)
            assert set(scored) == set(inview)
            assert all(np.isfinite(v) for v in scored.values())


test_score_inview_adapters()
print("ok: score_inview adapters have a uniform signature and no NaN/inf, every dataset")

ok: score_inview adapters have a uniform signature and no NaN/inf, every dataset


## generate_predictions

Per SPEC.md Q5 #2/#4:

- One line per impression: `{native_impression_id} [{rank_1},...,{rank_n}]`,
  `rank_i` the 1-indexed rank (1 = best) of the *i*-th article in that
  impression's `article_ids_inview`, in its **original** order.
- `native_impression_id` recovers the dataset's un-namespaced ID by taking
  the trailing token of `impression_id` (`mind_dev_1` -> `1`,
  `ebnerd_144772` -> `144772`).
- **Row order in the output file matches `behaviors.parquet`'s existing row
  order** (itself unmodified from the raw files -- Q1's pipeline never
  sorts/reorders), *not* the user-sorted order used below purely for the
  BM25 adapter's cache efficiency.
- Zip contains exactly one file (`TXT_FILENAME[dataset]`) at the zip root
  (no subfolder, no `__MACOSX/` entries).

In [5]:
def generate_predictions(dataset: str, method: str, split: str = SPLIT) -> Path:
    split_behaviors = feature_store[dataset]["behaviors"].with_row_index("row_idx")
    score_fn = score_inview_adapters[dataset][method]

    # Compute in a user-sorted order for the BM25 adapter's cache efficiency,
    # keyed by each row's original row index (synthetic, added above -- the
    # behaviors table was already read in the file's original row order via
    # the setup cell's scan_parquet, so row_idx recovers that order exactly)
    # so results can be written back out in the file's original order below.
    compute_order = split_behaviors.sort("user_id")
    row_idx_col = compute_order["row_idx"].to_list()
    user_id_col = compute_order["user_id"].to_list()
    inview_col = compute_order["article_ids_inview"].to_list()
    n_rows = len(row_idx_col)
    log_progress(f"  {dataset}/{method}: scoring {n_rows} {split} impressions")

    ranks_by_row = {}
    for i, (row_idx, user_id, inview) in enumerate(zip(row_idx_col, user_id_col, inview_col)):
        inview_ids = list(inview)
        scored = score_fn(user_id, inview_ids)
        scores = np.array([scored[aid] for aid in inview_ids])
        ranks = (np.argsort(np.argsort(-scores, kind="stable"), kind="stable") + 1).tolist()
        ranks_by_row[row_idx] = ranks
        if (i + 1) % 200_000 == 0:
            log_progress(f"    {dataset}/{method}: {i + 1}/{n_rows} impressions scored")

    lines = []
    for row_idx, impression_id in zip(split_behaviors["row_idx"].to_list(), split_behaviors["impression_id"].to_list()):
        native_id = impression_id.rsplit("_", 1)[-1]
        ranks_str = "[" + ",".join(str(r) for r in ranks_by_row[row_idx]) + "]"
        lines.append(f"{native_id} {ranks_str}")

    out_dir = SUBMISSIONS_DIR / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    txt_filename = TXT_FILENAME[dataset]
    txt_path = out_dir / txt_filename
    # write_text() would translate \n -> \r\n on Windows; write bytes
    # directly so the file matches the reference implementations exactly.
    txt_path.write_bytes(("\n".join(lines) + "\n").encode("utf-8"))

    zip_path = out_dir / f"{dataset}_{method}_{split}_predictions.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(txt_path, arcname=txt_filename)
    txt_path.unlink()
    log_progress(f"  {dataset}/{method}: wrote {zip_path.name} ({len(lines)} lines)")
    return zip_path


prediction_zips = {
    (name, method): generate_predictions(name, method) for name in DATASETS for method in METHODS
}
log_progress("generate_predictions: all zips written")
prediction_zips

{('ebnerd_large',
  'embedding'): WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/ebnerd_large/ebnerd_large_embedding_test_predictions.zip'),
 ('ebnerd_large',
  'bm25'): WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/ebnerd_large/ebnerd_large_bm25_test_predictions.zip'),
 ('mind_large',
  'embedding'): WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/mind_large/mind_large_embedding_test_predictions.zip'),
 ('mind_large',
  'bm25'): WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/mind_large/mind_large_bm25_test_predictions.zip')}

In [6]:
def test_generate_predictions():
    for name in DATASETS:
        split_behaviors = feature_store[name]["behaviors"]
        n_expected = split_behaviors.height
        expected_native_ids = [iid.rsplit("_", 1)[-1] for iid in split_behaviors["impression_id"].to_list()]
        expected_inview_lens = [len(x) for x in split_behaviors["article_ids_inview"].to_list()]

        for method in METHODS:
            zip_path = prediction_zips[(name, method)]
            assert zip_path.exists()

            txt_filename = TXT_FILENAME[name]
            with zipfile.ZipFile(zip_path) as zf:
                names = zf.namelist()
                assert names == [txt_filename], f"zip must contain exactly {txt_filename} at root, got {names}"
                content = zf.read(txt_filename).decode("utf-8")

            lines = content.strip("\n").split("\n")
            assert len(lines) == n_expected

            for line, expected_id, expected_len in zip(lines, expected_native_ids, expected_inview_lens):
                impr_id_str, ranks_str = line.split(" ", 1)
                assert impr_id_str == expected_id, "row order must match the original behaviors file order"
                ranks = [int(r) for r in ranks_str.strip("[]").split(",")]
                assert len(ranks) == expected_len
                assert sorted(ranks) == list(range(1, expected_len + 1)), "ranks must be a permutation starting at 1"


test_generate_predictions()
log_progress("generate_predictions completed successfully")
print("ok: prediction files round-trip for every dataset -- correct row count, row order, and valid 1..n rank permutations")

ok: prediction files round-trip for every dataset -- correct row count, row order, and valid 1..n rank permutations


# Manual Verification Complete